# Stage 5: Transformer Models

Here we fine-tune DistilBERT on our dataset and compare its performance to the baselines.

In [1]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

sys.path.append("../src")
from issue_intelligence.models.transformer import prepare_dataset, build_transformer_trainer

import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data/processed")


## 1. Load Data

In [2]:
def load_data(file_path):
    X, y = [], []
    if not file_path.exists():
        return X, y
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            X.append(r.get("combined_text", ""))
            y.append(r.get("target", "Unknown"))
    return X, y

X_train, y_train = load_data(DATA_DIR / "scikit-learn_issues_model_stratified_train.jsonl")
X_test, y_test = load_data(DATA_DIR / "scikit-learn_issues_model_stratified_test.jsonl")

le = LabelEncoder()
train_dataset, le = prepare_dataset(X_train, y_train, le)
test_dataset, le = prepare_dataset(X_test, y_test, le)

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")
print(f"Classes: {le.classes_}")


Train size: 69
Test size: 16
Classes: ['Bug' 'Documentation' 'Enhancement']


## 2. Train DistilBERT

In [3]:
trainer, tokenizer = build_transformer_trainer(
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    model_name="distilbert-base-uncased",
    num_labels=len(le.classes_),
    epochs=2,
    batch_size=8,
)

print("Starting Fine-Tuning...")
trainer.train()


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

...

Starting Fine-Tuning...


Epoch,Training Loss,Validation Loss,F1
1,No log,0.999820,0.222222
2,1.071994,0.867002,0.222222


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18, training_loss=1.0096533298492432, metrics={'train_runtime': 21.8922, 'train_samples_per_second': 6.304, 'train_steps_per_second': 0.822, 'total_flos': 4570206755328.0, 'train_loss': 1.0096533298492432, 'epoch': 2.0})

## 3. Evaluation

In [4]:
print("Evaluating on Test Set...")
eval_results = trainer.evaluate()
print(f"Transformer Macro F1: {eval_results['eval_f1']:.4f}")

# Get raw predictions
predictions = trainer.predict(trainer.eval_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

y_true_labels = le.inverse_transform(predictions.label_ids)
y_pred_labels = le.inverse_transform(preds)

print("\nClassification Report:\n")
print(classification_report(y_true_labels, y_pred_labels, zero_division=0))


Evaluating on Test Set...


Training Loss,Validation Loss,Epoch,F1
1.071994,0.867002,2,0.222222


Transformer Macro F1: 0.2222



Classification Report:

               precision    recall  f1-score   support

          Bug       0.50      1.00      0.67         8
Documentation       0.00      0.00      0.00         3
  Enhancement       0.00      0.00      0.00         5

     accuracy                           0.50        16
    macro avg       0.17      0.33      0.22        16
 weighted avg       0.25      0.50      0.33        16

